# 08. Final Analysis

This notebook consolidates the main results from the LST downscaling
workflow for Lucknow.

The final analysis summarizes the results of the TsHARP, Random Forest,
and XGBoost downscaling approaches using outputs generated in the
previous notebooks.

The analysis focuses on:

- machine-learning predictive performance,
- coarse-scale consistency,
- spatial behaviour,
- thermal-range characteristics,
- and model explainability.

The objective is to organize the final evidence from the complete
workflow and establish the main findings of the study.

In [36]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

PROJECT_DIR = Path(r"D:\Projects\GeoAI-LST-Downscaling")

FIGURES_DIR = PROJECT_DIR / "figures" / "lucknow"
RESULTS_DIR = PROJECT_DIR / "results" 

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Figures directory:", FIGURES_DIR)
print("Results directory:", RESULTS_DIR)

Project directory: D:\Projects\GeoAI-LST-Downscaling
Figures directory: D:\Projects\GeoAI-LST-Downscaling\figures\lucknow
Results directory: D:\Projects\GeoAI-LST-Downscaling\results


In [37]:
for path in RESULTS_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

results\30m_90m_consistency_comparison.csv
results\30m_product_statistics.csv
results\final_coarse_scale_comparison.csv
results\final_findings_summary.csv
results\final_ml_model_comparison.csv
results\final_product_statistics.csv
results\ml_model_performance.csv
results\spatial_difference_statistics.csv
results\thermal_range_comparison.csv


## 2. Inspect Existing Project Outputs

The previous notebooks generated the main visual outputs of the
downscaling workflow.

This section checks the existing figure files so that the final
analysis can reuse them rather than unnecessarily recalculating
results.

In [38]:
print("Existing figures:\n")

for path in sorted(FIGURES_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

Existing figures:

figures\lucknow\notebook_1\lst_celsius_lucknow.png
figures\lucknow\notebook_1\lst_clean_lucknow.png
figures\lucknow\notebook_1\lst_uncertainty_lucknow.png
figures\lucknow\notebook_1\lucknow_quality_mask.png
figures\lucknow\notebook_1\ndvi_lucknow.png
figures\lucknow\notebook_1\ndvi_vs_lst_lucknow.png
figures\lucknow\notebook_1\nir_lucknow_reflectance.png
figures\lucknow\notebook_1\red_lucknow_reflectance.png
figures\lucknow\notebook_2\green_lucknow_reflectance.png
figures\lucknow\notebook_2\mndwi_lucknow.png
figures\lucknow\notebook_2\ndbi_lucknow.png
figures\lucknow\notebook_2\swir1_lucknow_reflectance.png
figures\lucknow\notebook_2\swir2_lucknow_reflectance.png
figures\lucknow\notebook_3\tsharp_absolute_error_30m.png
figures\lucknow\notebook_3\tsharp_lst_30m.png
figures\lucknow\notebook_4\rf_consistency_30m.png
figures\lucknow\notebook_4\rf_feature_importance.png
figures\lucknow\notebook_4\rf_lst_30m_lucknow.png
figures\lucknow\notebook_4\rf_observed_vs_predicted.p

## 3. Final Model Performance Comparison

This section summarizes the final predictive performance of the
machine-learning downscaling models.

Random Forest and XGBoost are compared using the spatially withheld
test observations. The comparison focuses on MAE, RMSE, R², and bias.

The values are taken from the completed model-evaluation analysis and
are not recalculated in this notebook.

In [39]:
final_metrics = pd.read_csv(
    RESULTS_DIR / "ml_model_performance.csv"
)

display(final_metrics)

,Model,MAE,RMSE,R2
0,Random Forest,0.742,0.968,0.684
1,XGBoost,0.741,0.970,0.683


In [40]:
final_metrics.to_csv(
    RESULTS_DIR / "final_ml_model_comparison.csv",
    index=False
)

print("Final ML comparison saved.")

Final ML comparison saved.


## 4. Thermal Range Comparison

This section compares the thermal range represented by the original
90 m LST and the three 30 m downscaled LST products:

- Original 90 m LST
- TsHARP 30 m LST
- Random Forest 30 m LST
- XGBoost 30 m LST

The comparison examines the minimum, maximum, mean, standard deviation,
and thermal range of each product.

This analysis is used to determine how well each downscaling approach
preserves the observed thermal variability and whether the downscaled
products exhibit compression of thermal extremes.

In [41]:
lst_90m_path = PROJECT_DIR / "data" / "processed" / "lucknow" / "coarse" / "lst_90m_lucknow.tif"

tsharp_path = PROJECT_DIR / "data" / "processed" / "lucknow" / "downscaled" / "tsharp_lst_30m_lucknow.tif"

rf_path = PROJECT_DIR / "data" / "processed" / "lucknow" / "downscaled" / "rf_lst_30m_lucknow.tif"

xgb_path = PROJECT_DIR / "data" / "processed" / "lucknow" / "downscaled" / "xgb_lst_30m_lucknow.tif"

In [42]:
def get_raster_stats(path):
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)
        nodata = src.nodata

        if nodata is not None:
            data[data == nodata] = np.nan

    valid = data[np.isfinite(data)]

    return {
        "Min (°C)": np.min(valid),
        "Max (°C)": np.max(valid),
        "Mean (°C)": np.mean(valid),
        "Std Dev (°C)": np.std(valid),
        "Thermal Range (°C)": np.max(valid) - np.min(valid)
    }


thermal_stats = pd.DataFrame(
    {
        "Original 90 m": get_raster_stats(lst_90m_path),
        "TsHARP 30 m": get_raster_stats(tsharp_path),
        "RF 30 m": get_raster_stats(rf_path),
        "XGB 30 m": get_raster_stats(xgb_path)
    }
).T

display(thermal_stats)

,Min (°C),Max (°C),Mean (°C),Std Dev (°C),Thermal Range (°C)
Original 90 m,32.283291,44.784115,39.171636,1.490371,12.500824
TsHARP 30 m,30.975880,45.252350,39.176823,1.539358,14.276470
RF 30 m,33.090565,44.170734,39.220788,1.353049,11.080170
XGB 30 m,33.601151,44.467197,39.210473,1.326293,10.866047


In [43]:
thermal_stats.to_csv(
    RESULTS_DIR / "thermal_range_comparison.csv"
)

print("Thermal range comparison saved.")

Thermal range comparison saved.


In [44]:
print("Existing CSV result files:\n")

for path in sorted(RESULTS_DIR.rglob("*.csv")):
    print(path.relative_to(RESULTS_DIR))

Existing CSV result files:

30m_90m_consistency_comparison.csv
30m_product_statistics.csv
final_coarse_scale_comparison.csv
final_findings_summary.csv
final_ml_model_comparison.csv
final_product_statistics.csv
ml_model_performance.csv
spatial_difference_statistics.csv
thermal_range_comparison.csv


## 5. Consolidated Results

The main quantitative results generated in the previous notebooks are
combined in this section.

The existing CSV result files are reused rather than recalculating
model predictions or raster-based evaluation metrics.

The results cover:

- machine-learning model performance,
- coarse-scale consistency,
- thermal characteristics,
- spatial differences,
- and statistics of the downscaled LST products.

This provides a single location for reviewing the final quantitative
results of the study.

In [45]:
results_files = {
    "ML Performance": "ml_model_performance.csv",
    "Final ML Comparison": "final_ml_model_comparison.csv",
    "30m–90m Consistency": "30m_90m_consistency_comparison.csv",
    "Product Statistics": "30m_product_statistics.csv",
    "Spatial Differences": "spatial_difference_statistics.csv",
    "Thermal Range": "thermal_range_comparison.csv"
}

loaded_results = {}

for name, filename in results_files.items():

    path = RESULTS_DIR / filename

    if path.exists():
        loaded_results[name] = pd.read_csv(path)
        print(f"{name}: loaded")
    else:
        print(f"{name}: NOT FOUND")

ML Performance: loaded
Final ML Comparison: loaded
30m–90m Consistency: loaded
Product Statistics: loaded
Spatial Differences: loaded
Thermal Range: loaded


In [46]:
for name, table in loaded_results.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    display(table)


ML Performance


,Model,MAE,RMSE,R2
0,Random Forest,0.742,0.968,0.684
1,XGBoost,0.741,0.970,0.683



Final ML Comparison


,Model,MAE,RMSE,R2
0,Random Forest,0.742,0.968,0.684
1,XGBoost,0.741,0.970,0.683



30m–90m Consistency


...

,Model,Min,Max,Mean,Std
0,TsHARP,30.975880,45.252350,39.176820,1.539358
1,Random Forest,33.090565,44.170734,39.220787,1.353049
2,XGBoost,33.601150,44.467197,39.210472,1.326293



Spatial Differences


,Comparison,Mean Absolute Difference,Median Absolute Difference,Maximum Absolute Difference
0,RF - TsHARP,0.743877,0.584734,7.367474
1,XGBoost - TsHARP,0.736780,0.580910,7.066090
2,XGBoost - RF,0.174957,0.136547,2.803783



Thermal Range


,Unnamed: 0,Min (°C),Max (°C),Mean (°C),Std Dev (°C),Thermal Range (°C)
0,Original 90 m,32.283291,44.784115,39.171636,1.490371,12.500824
1,TsHARP 30 m,30.975880,45.252350,39.176823,1.539358,14.276470
2,RF 30 m,33.090565,44.170734,39.220788,1.353049,11.080170
3,XGB 30 m,33.601151,44.467197,39.210473,1.326293,10.866047


## 6. Final Model Comparison

The final model comparison combines the main evaluation results from
the Random Forest and XGBoost models with the coarse-scale consistency
results of the three downscaling approaches.

The machine-learning performance metrics describe the spatially
withheld test performance of Random Forest and XGBoost.

The coarse-scale consistency metrics provide a separate comparison
of TsHARP, Random Forest, and XGBoost against the original 90 m LST.

These two evaluations are kept separate because the machine-learning
test evaluation and the coarse-scale consistency assessment represent
different evaluation procedures.

In [47]:
for name, table in loaded_results.items():
    print(f"\n{name}")
    print("Columns:", list(table.columns))
    print("Shape:", table.shape)


ML Performance
Columns: ['Model', 'MAE', 'RMSE', 'R2']
Shape: (2, 4)

Final ML Comparison
Columns: ['Model', 'MAE', 'RMSE', 'R2']
Shape: (2, 4)

30m–90m Consistency
Columns: ['Model', 'MAE', 'RMSE', 'R2', 'Bias']
Shape: (3, 5)

Product Statistics
Columns: ['Model', 'Min', 'Max', 'Mean', 'Std']
Shape: (3, 5)

Spatial Differences
Columns: ['Comparison', 'Mean Absolute Difference', 'Median Absolute Difference', 'Maximum Absolute Difference']
Shape: (3, 4)

Thermal Range
Columns: ['Unnamed: 0', 'Min (°C)', 'Max (°C)', 'Mean (°C)', 'Std Dev (°C)', 'Thermal Range (°C)']
Shape: (4, 6)


In [48]:
ml_final = loaded_results["Final ML Comparison"].copy()
consistency_final = loaded_results["30m–90m Consistency"].copy()

print("Machine-learning test performance:")
display(ml_final)

print("Coarse-scale consistency:")
display(consistency_final)

Machine-learning test performance:


,Model,MAE,RMSE,R2
0,Random Forest,0.742,0.968,0.684
1,XGBoost,0.741,0.970,0.683


Coarse-scale consistency:


,Model,MAE,RMSE,R2,Bias
0,TsHARP,0.000002,0.000003,1.000000,3.018724e-08
1,Random Forest,0.637307,0.828625,0.690880,4.580944e-02
2,XGBoost,0.643180,0.836417,0.685039,3.570386e-02


### Final Comparison Tables

The final machine-learning performance and coarse-scale consistency
tables are saved separately so that the evaluation procedures remain
clearly distinguished.

In [49]:
ml_final.to_csv(
    RESULTS_DIR / "final_ml_model_comparison.csv",
    index=False
)

consistency_final.to_csv(
    RESULTS_DIR / "final_coarse_scale_comparison.csv",
    index=False
)

print("Final comparison tables saved.")

Final comparison tables saved.


## 7. Final Downscaled Product Statistics

This section summarizes the statistical characteristics of the three
30 m downscaled LST products.

The minimum, maximum, mean, and standard deviation are compared to
identify differences in the thermal distribution produced by TsHARP,
Random Forest, and XGBoost.

In [50]:
product_stats = loaded_results["Product Statistics"].copy()

display(product_stats)

,Model,Min,Max,Mean,Std
0,TsHARP,30.975880,45.252350,39.176820,1.539358
1,Random Forest,33.090565,44.170734,39.220787,1.353049
2,XGBoost,33.601150,44.467197,39.210472,1.326293


In [51]:
# Save final product statistics

product_stats.to_csv(
    RESULTS_DIR / "final_product_statistics.csv",
    index=False
)

print("Final product statistics saved.")

Final product statistics saved.


## 8. Final Findings

The final analysis brings together the predictive performance,
coarse-scale consistency, spatial differences, thermal characteristics,
and SHAP-based model interpretation obtained throughout the project.

The main findings are:

1. Random Forest and XGBoost show very similar spatial test
   performance, with neither model demonstrating a substantial
   predictive advantage over the other.

2. Both machine-learning models reproduce the overall mean thermal
   level closely, but their predicted thermal distributions show
   reduced variability compared with the original 90 m LST.

3. TsHARP produces a wider thermal range than the original 90 m LST,
   whereas Random Forest and XGBoost produce narrower thermal ranges,
   indicating greater compression of thermal extremes in the
   machine-learning products.

4. The coarse-scale consistency and spatial-difference results provide
   additional evidence of how each downscaling approach preserves the
   original thermal pattern.

5. SHAP analysis identifies SWIR2, NDBI, and NDVI as the dominant
   predictors for the machine-learning models.

6. Higher SWIR2 and NDBI values generally contribute positively to
   predicted LST, while higher NDVI values generally contribute
   negatively.

7. Although the two machine-learning models learn broadly similar
   relationships, they differ in the relative contribution of some
   secondary predictors.

Overall, the results show that Random Forest and XGBoost provide
similar predictive behaviour for the current study area and predictor
configuration. The analysis also demonstrates that model evaluation
should consider not only numerical predictive metrics but also
coarse-scale consistency, spatial behaviour, thermal variability,
and model interpretability.

In [52]:
final_findings = pd.DataFrame({
    "Finding": [
        "RF and XGBoost show similar spatial test performance.",
        "RF and XGBoost reproduce the overall mean thermal level closely.",
        "RF and XGBoost show narrower thermal distributions than the reference.",
        "TsHARP produces a wider thermal range than the reference.",
        "SWIR2, NDBI, and NDVI are the dominant SHAP predictors.",
        "Higher SWIR2 and NDBI generally contribute positively to predicted LST.",
        "Higher NDVI generally contributes negatively to predicted LST."
    ]
})

final_findings.to_csv(
    RESULTS_DIR / "final_findings_summary.csv",
    index=False
)

display(final_findings)

print("Final findings saved.")

,Finding
0,RF and XGBoost show similar spatial test perfo...
1,RF and XGBoost reproduce the overall mean ther...
2,RF and XGBoost show narrower thermal distribut...
3,TsHARP produces a wider thermal range than the...
4,"SWIR2, NDBI, and NDVI are the dominant SHAP pr..."
5,Higher SWIR2 and NDBI generally contribute pos...
6,Higher NDVI generally contributes negatively t...


Final findings saved.


## 9. Limitations

The current study has several limitations.

First, the SHAP analysis was performed using a 100-observation sample
because of the computational cost associated with explaining the
300-tree Random Forest model.

Second, the spatial test evaluation of Random Forest and XGBoost and
the TsHARP assessment use different validation procedures. Therefore,
their numerical performance metrics should not be interpreted as a
direct one-to-one comparison.

Third, the machine-learning models use a fixed set of remote-sensing
predictors and a single study area. The observed relationships may
therefore depend on the characteristics of the Lucknow study area,
the available imagery, and the selected predictor configuration.

Finally, SHAP describes the behaviour of the trained machine-learning
models and does not establish physical causation between individual
predictors and LST.